In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

from xgboost import XGBRegressor


In [ ]:
file_path = r"C:\Solar_SUDHA_maam\fixed_dataset.csv"
df = pd.read_csv(file_path)

df.columns = df.columns.str.strip().str.lower()

df["time"] = pd.to_datetime(df["time"], dayfirst=True)

df = df.sort_values("time").reset_index(drop=True)

df.head()


In [ ]:
df = df.dropna()
df = df[df["power"] >= 0]

print(df.shape)


In [ ]:
df["hour"] = df["time"].dt.hour + df["time"].dt.minute / 60
df["day_of_year"] = df["time"].dt.dayofyear
df["month"] = df["time"].dt.month

# Cyclic hour encoding
df["sin_hour"] = np.sin(2 * np.pi * df["hour"] / 24)
df["cos_hour"] = np.cos(2 * np.pi * df["hour"] / 24)

# Seasonal encoding
df["sin_doy"] = np.sin(2 * np.pi * df["day_of_year"] / 365)
df["cos_doy"] = np.cos(2 * np.pi * df["day_of_year"] / 365)


In [ ]:
df["temp_x_hour"] = df["temp"] * df["sin_hour"]
df["humidity_x_hour"] = df["humidity"] * df["sin_hour"]
df["wind_x_hour"] = df["wind_speed"] * df["sin_hour"]


In [ ]:
df["power_lag1"] = df["power"].shift(1)
df["power_lag2"] = df["power"].shift(2)

df = df.dropna()


In [ ]:
features = [
    "temp",
    "humidity",
    "wind_speed",
    "precipitation",
    "sin_hour",
    "cos_hour",
    "sin_doy",
    "cos_doy",
    "temp_x_hour",
    "humidity_x_hour",
    "wind_x_hour",
    "power_lag1",
    "power_lag2"
]

target = "power"


In [ ]:
split_date = "2022-07-01"

train = df[df["time"] < split_date]
test  = df[df["time"] >= split_date]

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
model = XGBRegressor(
    n_estimators=1200,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=2,
    reg_lambda=5,
    min_child_weight=5,
    gamma=1,
    random_state=42
)

model.fit(
    X_train_scaled,
    y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=False
)


In [ ]:
pred = model.predict(X_test_scaled)
pred = np.clip(pred, 0, None)


In [ ]:
print("R2:", r2_score(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))


In [ ]:
plt.figure(figsize=(14,6))
plt.plot(test["time"], y_test, label="Actual", linewidth=1)
plt.plot(test["time"], pred, label="Predicted", linewidth=1)
plt.legend()
plt.title("Actual vs Predicted Solar Power")
plt.show()


In [ ]:
def plot_day(date_string):
    selected = test[test["time"].dt.date == pd.to_datetime(date_string).date()]
    idx = selected.index

    plt.figure(figsize=(14,5))
    plt.plot(selected["time"], selected["power"], label="Actual")
    plt.plot(selected["time"], pred[idx - test.index[0]], label="Predicted")
    plt.legend()
    plt.title(f"Solar Power - {date_string}")
    plt.show()


In [ ]:
def predict_power(date_time, temp, humidity, wind_speed, precipitation, prev_power1, prev_power2):

    dt = pd.to_datetime(date_time)

    hour = dt.hour + dt.minute / 60
    day_of_year = dt.dayofyear

    sin_hour = np.sin(2 * np.pi * hour / 24)
    cos_hour = np.cos(2 * np.pi * hour / 24)

    sin_doy = np.sin(2 * np.pi * day_of_year / 365)
    cos_doy = np.cos(2 * np.pi * day_of_year / 365)

    temp_x_hour = temp * sin_hour
    humidity_x_hour = humidity * sin_hour
    wind_x_hour = wind_speed * sin_hour

    input_data = pd.DataFrame([[
        temp,
        humidity,
        wind_speed,
        precipitation,
        sin_hour,
        cos_hour,
        sin_doy,
        cos_doy,
        temp_x_hour,
        humidity_x_hour,
        wind_x_hour,
        prev_power1,
        prev_power2
    ]], columns=features)

    input_scaled = scaler.transform(input_data)

    prediction = model.predict(input_scaled)[0]

    return max(prediction, 0)


In [ ]:
print("R2:", r2_score(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))


In [ ]:
selected_date = "2022-08-20"

day_data = df[df["time"].dt.date == pd.to_datetime(selected_date).date()]
day_data = day_data.sort_values("time").reset_index(drop=True)

print("Rows in this day:", len(day_data))

predictions = []

# Initialize lags with 0 (or previous day's last values if available)
lag1 = 0
lag2 = 0

for i in range(len(day_data)):

    row = day_data.iloc[i]

    feature_row = [
        row["temp"],
        row["humidity"],
        row["wind_speed"],
        row["precipitation"],
        row["sin_hour"],
        row["cos_hour"],
        row["sin_doy"],
        row["cos_doy"],
        row["temp_x_hour"],
        row["humidity_x_hour"],
        row["wind_x_hour"],
        lag1,
        lag2
    ]

    X_input = np.array(feature_row).reshape(1, -1)
    X_input_scaled = scaler.transform(X_input)

    pred = model.predict(X_input_scaled)[0]
    pred = max(pred, 0)

    predictions.append(pred)

    # Update lags using predicted values (NOT actual)
    lag2 = lag1
    lag1 = pred

pred_day = np.array(predictions)

plt.figure(figsize=(12,6))

plt.plot(day_data["time"], day_data["power"], label="Actual", linewidth=2)
plt.plot(day_data["time"], pred_day, label="Predicted (Recursive)", linewidth=2)

plt.title(f"5-Minute Solar Power Forecast — {selected_date}")
plt.xlabel("Time")
plt.ylabel("Power (kW)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()
